# 02 - Code graph extraction


## Goal

Build the NetworkX graph from the extraction bundle, save it to JSON, and draw a tiny ego subgraph around `KFileSearcher` so you can *see* the ontology.


## Prerequisites

- Notebook 01 ran successfully (we re-build the bundle here so this notebook is independent).
- `matplotlib` available locally for the visualization cell. The cell skips gracefully if not.


## Environment bootstrap

This cell makes the notebook portable between a local checkout and Colab.

- **Local**: when the notebook lives inside the repo, we add the repo root to `sys.path`
  so the `src` package imports cleanly.
- **Colab**: the import will fail with `ModuleNotFoundError`. We catch that and print a
  one-line reminder showing the `git clone` the learner should run. We deliberately do
  **not** execute the clone for them — the lab policy is *recipes only, no auto-downloads*.


In [ ]:
import sys
from pathlib import Path

try:
    # Local checkout: walk up from the notebook to the repo root.
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "common" / "paths.py").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            break
    from src.common.paths import REPO_ROOT, MINI_REPO, ensure_dirs
    ensure_dirs()
    print(f"repo root: {REPO_ROOT}")
    print(f"mini repo: {MINI_REPO}")
except ModuleNotFoundError:
    print("`src` not importable. If you are on Colab, run this in a separate cell:")
    print("    !git clone https://example.invalid/kde_ontology_slm_lab.git")
    print("    %cd kde_ontology_slm_lab")
    print("Then re-run this cell. We will not auto-clone for you (lab policy:")
    print("recipes only, no auto-downloads).")


## 1. Rebuild the bundle

We re-run the extraction so this notebook is self-contained. The logic is the same as `examples/run_mini_repo_pipeline.py`.


In [ ]:
from src.repo_ingest.scanner import scan
from src.repo_ingest.cmake_reader import read_cmake
from src.repo_ingest.cpp_reader import read_cpp
from src.repo_ingest.qml_reader import read_qml
from src.repo_ingest.dbus_reader import read_dbus
from src.repo_ingest.kconfig_reader import read_kconfig
from src.repo_ingest.desktop_file_reader import read_desktop
from src.repo_ingest.log_reader import read_log
from src.ontology.extractor import (
    ExtractionBundle, from_cmake, from_cpp, from_qml, from_dbus,
    from_kconfig, from_desktop, from_log,
)
from src.ontology.schema import Entity
from src.common.ids import make_id
from src.common.paths import MINI_REPO, GRAPHS_DIR

report = scan(MINI_REPO)
bundle = ExtractionBundle()
repo_id = bundle.add_entity(Entity(
    id=make_id('Repository', MINI_REPO.name),
    type='Repository', name=MINI_REPO.name, source_path=str(MINI_REPO),
))
for sf in report.by_kind('cmake'):
    from_cmake(bundle, read_cmake(sf.path), repo_id)
for sf in report.by_kind('cpp_header') + report.by_kind('cpp_source'):
    from_cpp(bundle, read_cpp(sf.path))
for sf in report.by_kind('qml'):
    from_qml(bundle, read_qml(sf.path))
for sf in report.by_kind('dbus'):
    from_dbus(bundle, read_dbus(sf.path))
for sf in report.by_kind('kconfig'):
    from_kconfig(bundle, read_kconfig(sf.path))
for sf in report.by_kind('desktop'):
    from_desktop(bundle, read_desktop(sf.path))
for sf in report.by_kind('log'):
    from_log(bundle, read_log(sf.path))
print(f'entities={len(bundle.entities)} relations={len(bundle.relations)}')


## 2. Build the graph

`src/graph/builder.py` converts the bundle into a `networkx.MultiDiGraph`. Edge keys are the relation type, so a single `(u, v)` pair can carry several relations without collisions.


In [ ]:
from src.graph.builder import build_graph, save_json, save_graphml

g = build_graph(bundle)
print(f'nodes: {g.number_of_nodes()}')
print(f'edges: {g.number_of_edges()}')
print(f'density: {g.number_of_edges() / max(1, g.number_of_nodes()):.2f} edges/node')


## 3. Persist the graph

JSON is the canonical interchange format. GraphML is convenient for Gephi or yEd. Both go under `artifacts/graphs/`.


In [ ]:
json_path = save_json(g, GRAPHS_DIR / 'mini_repo.json')
graphml_path = save_graphml(g, GRAPHS_DIR / 'mini_repo.graphml')
print('json    :', json_path)
print('graphml :', graphml_path)


## 4. Find KFileSearcher and pull its ego subgraph

We use `find_by_name` (see `src/graph/queries.py`) for case-insensitive lookup. `nx.ego_graph` gives us KFileSearcher's 1-hop neighbourhood.


In [ ]:
import networkx as nx
from src.graph.queries import find_by_name

matches = find_by_name(g, 'KFileSearcher', types={'CppClass'})
assert matches, 'KFileSearcher not found; did extraction run?'
center = matches[0]
ego = nx.ego_graph(g, center, radius=1, undirected=True)
print(f'ego nodes: {ego.number_of_nodes()}, edges: {ego.number_of_edges()}')
for n, d in ego.nodes(data=True):
    print(f"  {d.get('type','?'):14s} {d.get('name','')}")


## 5. Visualize (matplotlib, lazy import)

We import `matplotlib` only when we are about to draw — keeping the rest of the lab headless-friendly. If `matplotlib` is missing, the cell prints a hint and continues.


In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print('matplotlib not installed; skipping plot. `pip install matplotlib` to enable.')
else:
    pos = nx.spring_layout(ego, seed=42)
    color_by_type = {
        'CppClass': '#4c72b0', 'Signal': '#dd8452', 'Slot': '#55a868',
        'ConfigKey': '#c44e52', 'LogCategory': '#8172b3', 'HeaderFile': '#937860',
        'SourceFile': '#da8bc3', 'QmlComponent': '#8c8c8c',
    }
    colors = [color_by_type.get(d.get('type'), '#cccccc') for _, d in ego.nodes(data=True)]
    labels = {n: d.get('name', n)[:20] for n, d in ego.nodes(data=True)}
    plt.figure(figsize=(10, 7))
    nx.draw_networkx_nodes(ego, pos, node_color=colors, node_size=600)
    nx.draw_networkx_edges(ego, pos, alpha=0.4, arrows=True)
    nx.draw_networkx_labels(ego, pos, labels=labels, font_size=8)
    plt.title('Ego subgraph around KFileSearcher (radius=1)')
    plt.axis('off')
    plt.tight_layout()
    plt.show()


## Summary

You built a NetworkX graph, persisted it to JSON and GraphML, and drew a small ego subgraph around `KFileSearcher`. Notebook 03 zooms further into one slice: how QML components connect to their C++ backends.


## Exercises

1. Plot a *radius=2* ego graph instead of 1. What new entity types appear?
2. Color-code edges by relation type (HINT: `g.get_edge_data(u, v)` returns a dict keyed by `key`).
3. Export the JSON, open `artifacts/graphs/mini_repo.json` in a text editor, and identify the single longest path from the `Repository` node.
